# BDT Plotting

# Imports

In [ ]:
import sys
sys.path.insert(0, 'backend_functions')

import importlib
import selection_functions as sf
importlib.reload(sf)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import pickle
from pathlib import Path

print("✅ Imports complete")

# Configuration & Load Saved Files

In [ ]:
# ── only thing to set manually ───────────────────────────────────────────────
HORN_CURRENT = 'RHC'   # 'FHC' or 'RHC'
# ─────────────────────────────────────────────────────────────────────────────

# ── locate files ─────────────────────────────────────────────────────────────
def _latest(pattern):
    """Return the newest file matching a glob pattern, or raise."""
    candidates = sorted(Path('.').glob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No file found matching: {pattern}")
    if len(candidates) > 1:
        print(f"⚠️  Multiple matches for '{pattern}', using newest: {candidates[-1]}")
    return candidates[-1]

# ── load training info (recovers all config) ──────────────────────────────────
info_candidates = sorted(Path('BDT_models').glob(f'bdt_*_{HORN_CURRENT}*_info.pkl'))
if not info_candidates:
    raise FileNotFoundError(f"No _info.pkl found in BDT_models/ for {HORN_CURRENT}.")
info_path = info_candidates[-1]
with open(info_path, 'rb') as f:
    training_info = pickle.load(f)
print(f"✅ Loaded training info from: {info_path}")

RUN_PERIODS       = training_info['run_periods']
USE_EXT_IN_BDT    = training_info['use_ext_in_bdt']
ISRUN3            = training_info['isrun3']
ROUNDS            = training_info['rounds']
TRAIN_TEST_SPLIT  = training_info['train_test_split']
TRAIN_QUERY       = training_info['train_query']
TEST_QUERY        = training_info['test_query']
training_parameters = training_info['training_variables']

run_names  = '_'.join([k for k, v in RUN_PERIODS.items() if v])
ext_suffix = '_noext' if not USE_EXT_IN_BDT else '_withext'
horn_suffix = f'_{HORN_CURRENT}'

print(f"   Horn current:        {HORN_CURRENT}")
print(f"   Run periods:         {[k for k, v in RUN_PERIODS.items() if v]}")
print(f"   Boosting rounds:     {ROUNDS}")
print(f"   Training variables:  {len(training_parameters)}")

# ── load BDT metrics DataFrame ────────────────────────────────────────────────
metrics_path = _latest(f'bdt_metrics_df_{HORN_CURRENT}_*.pkl')
if not metrics_path.exists():
    metrics_path = Path(f'bdt_metrics_df_{HORN_CURRENT}_today.pkl')
bdt_metrics_df = pd.read_pickle(metrics_path)
print(f"✅ Loaded metrics from: {metrics_path}")

# ── load trained BDT model ────────────────────────────────────────────────────
bdt_model = xgb.Booster()
model_path = _latest(f'BDT_models/bdt_{run_names}{horn_suffix}{ext_suffix}*.model')
bdt_model.load_model(str(model_path))
print(f"✅ Loaded model from:   {model_path}")

# ── load training / test DataFrames ───────────────────────────────────────────
data_path = _latest(f'BDT_training_data/training_data_{run_names}_multirun{ext_suffix}*.pkl')
with open(data_path, 'rb') as f:
    training_data = pickle.load(f)
df_pre_train = training_data['df_pre_train']
df_pre_test  = training_data['df_pre_test']
df_pre       = training_data['df_pre']
print(f"✅ Loaded training data from: {data_path}")
print(f"   Train: {len(df_pre_train):,}  |  Test: {len(df_pre_test):,}")

# AUC / AUCPR Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ax.plot(bdt_metrics_df['train-auc'],  label='Train AUC',   color='#44863E', linestyle='-',  linewidth=2)
ax.plot(bdt_metrics_df['test-auc'],   label='Test AUC',    color='#44863E', linestyle='--', linewidth=2)
ax.plot(bdt_metrics_df['train-aucpr'], label='Train AUCPR', color='#9C3C75', linestyle='-',  linewidth=2)
ax.plot(bdt_metrics_df['test-aucpr'],  label='Test AUCPR',  color='#9C3C75', linestyle='--', linewidth=2)

ax.axvline(ROUNDS, color='gray', linestyle=':', linewidth=2, label=f'ROUNDS = {ROUNDS}')

ax.set_xlabel('Boosting Round', fontsize=16)
ax.set_ylabel('Score', fontsize=16)
ax.set_title(f'AUC and AUCPR vs Boosting Round — {HORN_CURRENT}', fontsize=17)
ax.tick_params(labelsize=15)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
ax.set_xlim(0, len(bdt_metrics_df) - 1)

plt.tight_layout()
out = f'bdt_training_curves_{HORN_CURRENT}.pdf'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f"✅ Saved to: {out}")

# BDT Selection Performance

In [ ]:
# Score BDT on test set if not already done
if 'BDT_score' not in df_pre_test.columns:
    dtest = xgb.DMatrix(df_pre_test[training_parameters])
    df_pre_test['BDT_score'] = bdt_model.predict(dtest)

# Scan score cuts
x = np.arange(0, 0.8, 0.025)

gen_data      = df_pre.query('is_signal == True').weight.values
gen_intrinsic = np.ones(len(gen_data))

perf_dict  = sf.bdt_pe(df_pre_test, x, gen_data, gen_intrinsic, test_size=TRAIN_TEST_SPLIT)
pur_bdt    = perf_dict['purity']
pur_err_bdt = perf_dict['purErr']
eff_bdt    = perf_dict['eff']
eff_err_bdt = perf_dict['effErr']

pur_bdt_arr = np.array(pur_bdt)
eff_bdt_arr = np.array(eff_bdt)
eff_pur     = eff_bdt_arr * pur_bdt_arr / 100

# Best cut: eff×pur with purity ≥ 75 %
mask = pur_bdt_arr >= 75
if np.any(mask):
    best_idx = np.argmax(eff_pur * mask)
else:
    best_idx = np.argmax(pur_bdt_arr)
best_cut = x[best_idx]

print(f"Best cut (purity ≥ 75%): BDT score > {best_cut:.3f}")
print(f"  Purity:    {pur_bdt[best_idx]:.2f}% ± {pur_err_bdt[best_idx]:.2f}%")
print(f"  Efficiency:{eff_bdt[best_idx]:.2f}% ± {eff_err_bdt[best_idx]:.2f}%")
print(f"  Eff×Pur:   {eff_pur[best_idx]:.2f}%")

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(x, eff_bdt, label='Efficiency',          color='#1f77b4', linewidth=2)
ax.plot(x, pur_bdt, label='Purity',              color='#ff7f0e', linewidth=2)
ax.plot(x, eff_pur, label='Efficiency × Purity', color='#2ca02c', linewidth=2)
ax.fill_between(x, eff_bdt_arr - eff_err_bdt, eff_bdt_arr + eff_err_bdt, color='#1f77b4', alpha=0.15)
ax.fill_between(x, pur_bdt_arr - pur_err_bdt, pur_bdt_arr + pur_err_bdt, color='#ff7f0e', alpha=0.15)
ax.axvline(best_cut, color='gray', linestyle=':', linewidth=2)
ax.axhline(75,       color='red',  linestyle='--', linewidth=2)
ax.set_xlabel('BDT Score Cut', fontsize=16)
ax.set_ylabel('Percentage',    fontsize=16)
ax.set_title(f'{HORN_CURRENT} Multi-Run BDT Selection Performance', fontsize=17)
ax.legend(fontsize=12)
ax.tick_params(labelsize=15)
ax.grid(alpha=0.3)
ax.set_xlim(x[0], x[-1])
plt.tight_layout()

out = f'bdt_performance_{run_names}_{HORN_CURRENT}{ext_suffix}_pur75.pdf'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f"✅ Saved to: {out}")

## Numerical Performance Table

In [ ]:
score_points  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for score in score_points:
    idx = np.argmin(np.abs(x - score))
    ep  = pur_bdt[idx] * eff_bdt[idx] / 100
    rows.append({
        'BDT Score >':       f'{x[idx]:.2f}',
        'Purity (%)':        f'{pur_bdt[idx]:.1f}',
        'Efficiency (%)':    f'{eff_bdt[idx]:.1f}',
        'Eff × Pur':         f'{ep:.1f}',
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

# Best Eff×Pur across scanned points
best_row = max(rows, key=lambda r: float(r['Eff × Pur']))
print(f"\nBest Eff×Pur: {best_row['Eff × Pur']}% at BDT score > {best_row['BDT Score >']}")

# SHAP Beeswarm Plot

In [ ]:
# df_pre_test was already cleaned during training (sentinels/overflows → NaN),
# and NaN values were handled natively by XGBoost during training.
# No further cleaning needed here.
print("🔬 Setting up SHAP analysis...")

X_test = df_pre_test[training_parameters].copy()

explainer = shap.TreeExplainer(bdt_model)

print(f"✅ SHAP explainer ready — {len(X_test):,} test samples, {len(training_parameters)} features")

In [ ]:
SHAP_SAMPLES = min(277005, len(X_test))
X_sample = X_test.sample(n=SHAP_SAMPLES, random_state=42)

print(f"Computing SHAP values for {SHAP_SAMPLES:,} samples...")
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(12, 8))
shap.plots.beeswarm(
    shap.Explanation(
        values=shap_values if not isinstance(shap_values, list) else shap_values[0],
        data=X_sample.values,
        feature_names=training_parameters
    ),
    max_display=15,
    show=False
)

ax = plt.gca()
for label in ax.get_yticklabels():
    label.set_fontfamily('monospace')
    label.set_fontsize(11)

plt.xlabel("SHAP Value (Impact on Model Output)", fontsize=12)
axes = plt.gcf().get_axes()
if len(axes) > 1:
    axes[-1].set_ylabel("Feature Value", fontsize=12)

plt.title(f'SHAP Beeswarm — {HORN_CURRENT} Multi-Run BDT', fontsize=14)

x_min, x_max = ax.get_xlim()
y_pos = ax.get_ylim()[0] + 0.5
ax.text(x_min, y_pos, "← Background-Like", fontsize=11, ha='left',  va='top', color='red')
ax.text(x_max, y_pos, "Signal-Like →",      fontsize=11, ha='right', va='top', color='darkgreen')

plt.tight_layout()
out = f'shap_beeswarm_{run_names}_{HORN_CURRENT}{ext_suffix}.pdf'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f"✅ Saved to: {out}")